In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

df_churn = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [2]:
# แปลงช่องว่าง ' ' เป็น NaN และเติมด้วย 0
df_churn['TotalCharges'] = pd.to_numeric(df_churn['TotalCharges'], errors='coerce')
df_churn['TotalCharges'] = df_churn['TotalCharges'].fillna(0)

In [3]:
# เลือก Features หลัก
X_churn = df_churn[['tenure', 'MonthlyCharges', 'TotalCharges']]
y_churn = df_churn['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

scaler_ch = StandardScaler()
X_scaled = scaler_ch.fit_transform(X_churn)

model_nn = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu', solver='adam', max_iter=500, random_state=42)

model_nn.fit(X_scaled, y_churn)

with open('../models/churn_model.pkl', 'wb') as f:
    pickle.dump(model_nn, f)

with open('../models/scaler_churn.pkl', 'wb') as f:
    pickle.dump(scaler_ch, f)

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# แบ่งข้อมูลก่อน scale
X_train, X_test, y_train, y_test = train_test_split(
    X_churn, y_churn,
    test_size=0.2,
    random_state=42,
    stratify=y_churn
)

# scale ใหม่
scaler_temp = StandardScaler()
X_train_scaled = scaler_temp.fit_transform(X_train)
X_test_scaled = scaler_temp.transform(X_test)

# สร้าง model ใหม่สำหรับ test
model_temp = MLPClassifier(
    hidden_layer_sizes=(16,8),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)

model_temp.fit(X_train_scaled, y_train)

y_pred = model_temp.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)

print(f"Test Accuracy: {acc*100:.2f}%")

Test Accuracy: 78.28%
